# 04 · Recomendación de reasignación de transportistas por zona
**Objetivo:** proponer una nueva asignación de transportistas por zona que mejore la puntualidad (OTD), sustentada en los indicadores calculados.

**Entrada:** `bd/courier_dw.sqlite` (modelo estrella generado por `02_etl.ipynb`)

**Método:** se usa el OTD histórico de cada combinación zona × transportista para estimar el OTD esperado de cualquier asignación.

## 1. Matriz de OTD y volumen actual

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from scipy.optimize import linprog

con = sqlite3.connect("bd/courier_dw.sqlite")
base = pd.read_sql("""
SELECT z.zona, t.codigo_transportista AS transportista,
       COUNT(*) AS envios, SUM(f.flag_a_tiempo) AS a_tiempo, SUM(f.flag_entregado) AS entregados
FROM fact_envios f
JOIN dim_zona z          ON z.id_zona = f.id_zona
JOIN dim_transportista t ON t.id_transportista = f.id_transportista
GROUP BY z.zona, t.codigo_transportista
""", con)
otd = (base.pivot(index="zona", columns="transportista", values="a_tiempo")
       / base.pivot(index="zona", columns="transportista", values="entregados"))
actual = base.pivot(index="zona", columns="transportista", values="envios")
orden = ["Norte", "Centro", "Sur", "Valles", "Rural"]
otd, actual = otd.loc[orden], actual.loc[orden]
(otd * 100).round(1)

transportista,TR01,TR02,TR03,TR04,TR05,TR06,TR07,TR08
zona,,,,,,,,
Norte,91.3,91.1,91.7,88.1,93.5,93.1,60.3,92.0
Centro,89.3,92.0,89.0,84.2,94.3,91.1,59.1,89.4
Sur,83.8,85.2,84.3,78.3,87.3,87.1,45.8,87.9
Valles,61.3,67.3,63.1,54.5,68.9,66.1,21.4,64.4
Rural,18.6,27.3,17.8,17.3,25.4,22.5,2.8,22.8


## 2. La asignación actual es pareja (asignación ineficiente)

In [2]:
participacion = actual.div(actual.sum(axis=1), axis=0) * 100
participacion.round(1)

transportista,TR01,TR02,TR03,TR04,TR05,TR06,TR07,TR08
zona,,,,,,,,
Norte,12.2,12.3,12.6,12.7,12.4,12.4,12.6,12.8
Centro,12.7,12.4,12.9,12.4,12.7,12.1,12.1,12.6
Sur,12.4,12.2,12.9,12.7,12.2,12.8,12.6,12.3
Valles,12.2,12.7,12.1,12.5,12.6,13.1,12.9,11.9
Rural,12.3,13.3,11.8,12.6,12.7,12.3,12.4,12.5


Todas las celdas rondan el 12,5 %: las rutas se reparten por igual sin considerar el desempeño.

## 3. Función de OTD esperado

In [3]:
def otd_esperado(asignacion):
    """OTD esperado = Σ (envíos asignados × OTD histórico de la celda) / envíos totales."""
    por_zona = (asignacion * otd).sum(axis=1) / asignacion.sum(axis=1)
    global_ = (asignacion * otd).values.sum() / asignacion.values.sum()
    return por_zona, global_

zona_actual, global_actual = otd_esperado(actual)
print(f"OTD esperado con la asignación actual: {global_actual:.1%}")

OTD esperado con la asignación actual: 74.5%


## 4. Propuesta por reglas

In [4]:
demanda = actual.sum(axis=1)
volumen = actual.sum(axis=0)
propuesta = pd.DataFrame(0.0, index=otd.index, columns=otd.columns)

# REGLA 1: Rural y Valles solo con los 4 mejores transportistas de cada zona
for zona in ["Rural", "Valles"]:
    mejores = otd.loc[zona].drop("TR07").sort_values(ascending=False).index[:4]
    print(f"{zona}: {list(mejores)}")
    propuesta.loc[zona, mejores] = demanda[zona] / 4

# REGLA 2: TR07 al 50 % de su volumen, solo en Norte y Centro
vol_tr07 = volumen["TR07"] * 0.5
propuesta.loc["Norte", "TR07"] = vol_tr07 / 2
propuesta.loc["Centro", "TR07"] = vol_tr07 / 2

# REGLA 3: el resto de Norte, Centro y Sur según la capacidad libre
otros = [t for t in otd.columns if t != "TR07"]
capacidad = volumen.mean() + (volumen["TR07"] - vol_tr07) / 7
libre = capacidad - propuesta[otros].sum(axis=0)
for zona in ["Norte", "Centro", "Sur"]:
    resto = demanda[zona] - propuesta.loc[zona].sum()
    propuesta.loc[zona, otros] = resto * libre / libre.sum()

propuesta = propuesta.round().astype(int)
propuesta

Rural: ['TR02', 'TR05', 'TR08', 'TR06']
Valles: ['TR05', 'TR02', 'TR06', 'TR08']


transportista,TR01,TR02,TR03,TR04,TR05,TR06,TR07,TR08
zona,,,,,,,,
Norte,2027,1116,2027,2027,1116,1116,1313,1116
Centro,1653,910,1653,1653,910,910,1313,910
Sur,1938,1067,1938,1938,1067,1067,0,1067
Valles,0,1488,0,0,1488,1488,0,1488
Rural,0,1036,0,0,1036,1036,0,1036


## 5. ¿Es realizable?

In [5]:
print(pd.DataFrame({"actual": volumen, "propuesto": propuesta.sum(axis=0)}))
print("\nDiferencia máxima de envíos por zona (redondeo):", (propuesta.sum(axis=1) - demanda).abs().max())

               actual  propuesto
transportista                   
TR01             5191       5618
TR02             5231       5617
TR03             5283       5618
TR04             5281       5618
TR05             5244       5617
TR06             5249       5617
TR07             5252       2626
TR08             5221       5617

Diferencia máxima de envíos por zona (redondeo): 2


## 6. OTD actual frente al propuesto

In [6]:
zona_prop, global_prop = otd_esperado(propuesta)
comparacion = pd.DataFrame({"OTD actual %": (zona_actual * 100).round(1),
                            "OTD propuesto %": (zona_prop * 100).round(1)})
comparacion["Cambio (pp)"] = (comparacion["OTD propuesto %"] - comparacion["OTD actual %"]).round(1)
print(comparacion)
print(f"\nOTD global: {global_actual:.1%}  →  {global_prop:.1%}")

        OTD actual %  OTD propuesto %  Cambio (pp)
zona                                              
Norte           87.6             87.8          0.2
Centro          86.2             85.3         -0.9
Sur             79.9             84.1          4.2
Valles          58.2             66.7          8.5
Rural           19.4             24.5          5.1

OTD global: 74.5%  →  77.1%


## 7. Validación con programación lineal (techo de referencia)

In [7]:
Z, T = otd.shape
cap = volumen * 1.12
cap["TR07"] = volumen["TR07"] * 0.5
limites = [(0, 0) if (tr == "TR07" and zona in ("Rural", "Valles")) else (0, 0.35 * demanda[zona])
           for zona in otd.index for tr in otd.columns]
r = linprog(c=-otd.values.ravel(),
            A_ub=np.kron(np.ones(Z), np.eye(T)), b_ub=cap.values,
            A_eq=np.kron(np.eye(Z), np.ones(T)), b_eq=demanda.values,
            bounds=limites, method="highs")
optimo = pd.DataFrame(r.x.reshape(Z, T), index=otd.index, columns=otd.columns).round().astype(int)
print("¿Encontró solución?", r.status == 0)
print(f"OTD máximo posible: {otd_esperado(optimo)[1]:.1%}")

¿Encontró solución? True
OTD máximo posible: 79.2%


## 8. Escenario: ajustar la promesa de la zona Rural

In [8]:
rural = pd.read_sql("""
SELECT f.dias_retraso FROM fact_envios f JOIN dim_zona z ON z.id_zona = f.id_zona
WHERE z.zona = 'Rural' AND f.flag_entregado = 1
""", con)
for extra in [0, 1, 2]:
    print(f"Promesa Estándar de {3 + extra} días → OTD Rural: {(rural['dias_retraso'] <= extra).mean():.1%}")

Promesa Estándar de 3 días → OTD Rural: 19.4%
Promesa Estándar de 4 días → OTD Rural: 47.7%
Promesa Estándar de 5 días → OTD Rural: 77.1%


## Recomendación
1. **Rural y Valles** solo con TR02, TR05, TR06 y TR08 (25 % cada uno).
2. **TR07** al 50 % de su volumen, solo en Norte y Centro, con un plan de mejora.
3. **Norte, Centro y Sur** repartidos entre los demás según su capacidad (+7 % cada uno).
4. **Promesa Rural Estándar de 5 días** (OTD Rural 19 % → 77 %).
5. **Reforzar el despacho de los lunes.**

**Impacto esperado:** OTD global 74,5 % → 77,1 % sin capacidad adicional (óptimo matemático: 79,2 %).

**Supuesto:** cada transportista mantiene su desempeño por zona al cambiar su volumen; se recomienda un piloto de 4 semanas.